# GCS Hadoop Catalog + Cloud SQL Iceberg Exploration

Use this notebook when you want one Spark session that can:

- read and write Iceberg tables through a GCS-backed Hadoop catalog for exploration
- read and write native Spark file formats on GCS: Parquet, CSV, JSON, and ORC
- optionally query the production Cloud SQL/JDBC Iceberg catalog when Cloud SQL Auth Proxy is running

The GCS Hadoop catalog path is best for exploration and sandbox tables. Keep production writes on the production catalog unless you intentionally migrate the pipeline.


## 1. Configuration

Defaults assume the local service account key is at `/tmp/alpaca-spark-gcs-reader.json` and the shaded GCS connector is at `/tmp/gcs-connector-hadoop3-2.2.30-shaded.jar`.

Write examples are disabled by default. Flip the `RUN_*_WRITE_SMOKE` flags only when you want to create sandbox objects in GCS.


In [ ]:
import os
import socket
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "project-66783f65-9c3e-4880-9a3")
BUCKET = os.environ.get("GCS_ICEBERG_BUCKET", f"{PROJECT_ID}-alpaca-iceberg-warehouse")
WAREHOUSE = os.environ.get("GCS_HADOOP_WAREHOUSE", f"gs://{BUCKET}")

GCS_CONNECTOR_JAR = os.environ.get("GCS_CONNECTOR_JAR", "/tmp/gcs-connector-hadoop3-2.2.30-shaded.jar")
GCS_SERVICE_ACCOUNT_JSON = os.environ.get("GCS_SERVICE_ACCOUNT_JSON", "/tmp/alpaca-spark-gcs-reader.json")

GCS_ICEBERG_CATALOG = os.environ.get("GCS_ICEBERG_CATALOG", "gcs_iceberg")
PROD_NAMESPACE = os.environ.get("ICEBERG_NAMESPACE", "alpaca")
PROD_TABLE = os.environ.get("ICEBERG_TABLE", "bars")
PROD_TABLE_ID = f"{GCS_ICEBERG_CATALOG}.{PROD_NAMESPACE}.{PROD_TABLE}"

EXPLORE_NAMESPACE = os.environ.get("GCS_EXPLORE_NAMESPACE", "explore")
EXPLORE_ICEBERG_TABLE = f"{GCS_ICEBERG_CATALOG}.{EXPLORE_NAMESPACE}.sample_bars"
EXPLORE_FILE_ROOT = os.environ.get("GCS_EXPLORE_FILE_ROOT", f"gs://{BUCKET}/explore/native_files")

# Safe defaults: reads are enabled, writes are opt-in.
RUN_HADOOP_ICEBERG_READ = os.environ.get("RUN_HADOOP_ICEBERG_READ", "true").lower() in {"1", "true", "yes"}
RUN_HADOOP_ICEBERG_WRITE_SMOKE = os.environ.get("RUN_HADOOP_ICEBERG_WRITE_SMOKE", "false").lower() in {"1", "true", "yes"}
RUN_NATIVE_FILE_WRITE_SMOKE = os.environ.get("RUN_NATIVE_FILE_WRITE_SMOKE", "false").lower() in {"1", "true", "yes"}
RUN_CLOUDSQL_CATALOG = os.environ.get("RUN_CLOUDSQL_CATALOG", "false").lower() in {"1", "true", "yes"}

CLOUDSQL_INSTANCE_CONNECTION_NAME = os.environ.get(
    "CLOUDSQL_INSTANCE_CONNECTION_NAME",
    "project-66783f65-9c3e-4880-9a3:us-east1:alpaca-iceberg-catalog",
)
CLOUDSQL_HOST = os.environ.get("CLOUDSQL_HOST", "127.0.0.1")
CLOUDSQL_PORT = int(os.environ.get("CLOUDSQL_PORT", "5432"))
CLOUDSQL_DB = os.environ.get("CLOUDSQL_DB", "iceberg")
CLOUDSQL_USER = os.environ.get("CLOUDSQL_USER", "iceberg")
CLOUDSQL_PASSWORD = os.environ.get("CLOUDSQL_PASSWORD")
CLOUDSQL_SECRET_NAME = os.environ.get("CLOUDSQL_SECRET_NAME", "ICEBERG_DB_PASSWORD")
CLOUDSQL_CATALOG = os.environ.get("CLOUDSQL_CATALOG", "cloudsql_iceberg")
CLOUDSQL_TABLE_ID = f"{CLOUDSQL_CATALOG}.{PROD_NAMESPACE}.{PROD_TABLE}"

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

print(f"Project              : {PROJECT_ID}")
print(f"Warehouse            : {WAREHOUSE}")
print(f"GCS Hadoop catalog   : {GCS_ICEBERG_CATALOG}")
print(f"Production table path: {PROD_TABLE_ID}")
print(f"Explore table        : {EXPLORE_ICEBERG_TABLE}")
print(f"Native file root     : {EXPLORE_FILE_ROOT}")
print(f"GCS key configured   : {Path(GCS_SERVICE_ACCOUNT_JSON).exists()} ({GCS_SERVICE_ACCOUNT_JSON})")
print(f"GCS connector exists : {Path(GCS_CONNECTOR_JAR).exists()} ({GCS_CONNECTOR_JAR})")


## 2. Start Spark

This Spark session registers two possible Iceberg catalogs:

- `gcs_iceberg`: Hadoop catalog backed by the GCS warehouse
- `cloudsql_iceberg`: optional JDBC catalog backed by Cloud SQL

The Cloud SQL catalog is only configured when `RUN_CLOUDSQL_CATALOG = True`.


In [ ]:
if not Path(GCS_CONNECTOR_JAR).is_file():
    raise RuntimeError(f"Missing GCS connector jar: {GCS_CONNECTOR_JAR}")
if not Path(GCS_SERVICE_ACCOUNT_JSON).is_file():
    raise RuntimeError(f"Missing service account key: {GCS_SERVICE_ACCOUNT_JSON}")

SPARK_PACKAGES = os.environ.get(
    "SPARK_PACKAGES",
    ",".join([
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2",
        "org.postgresql:postgresql:42.7.7",
    ]),
)

builder = (
    SparkSession.builder
    .master(os.environ.get("SPARK_MASTER", "local[*]"))
    .appName("alpaca-gcs-hadoop-catalog-explore")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.caseSensitive", "true")
    .config("spark.ui.enabled", os.environ.get("SPARK_UI_ENABLED", "false"))
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
    .config("spark.jars", GCS_CONNECTOR_JAR)
    .config("spark.jars.packages", SPARK_PACKAGES)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.fs.gs.project.id", PROJECT_ID)
    .config("spark.hadoop.fs.gs.auth.type", "SERVICE_ACCOUNT_JSON_KEYFILE")
    .config("spark.hadoop.fs.gs.auth.service.account.enable", "true")
    .config("spark.hadoop.fs.gs.auth.service.account.json.keyfile", GCS_SERVICE_ACCOUNT_JSON)
    .config(f"spark.sql.catalog.{GCS_ICEBERG_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{GCS_ICEBERG_CATALOG}.type", "hadoop")
    .config(f"spark.sql.catalog.{GCS_ICEBERG_CATALOG}.warehouse", WAREHOUSE)
)


def gcloud(*args: str) -> str:
    return subprocess.check_output(["gcloud", *args], text=True).strip()


def ensure_cloudsql_password() -> str:
    if CLOUDSQL_PASSWORD:
        return CLOUDSQL_PASSWORD
    return gcloud("secrets", "versions", "access", "latest", f"--secret={CLOUDSQL_SECRET_NAME}")


def preflight_cloudsql() -> None:
    try:
        with socket.create_connection((CLOUDSQL_HOST, CLOUDSQL_PORT), timeout=3):
            return
    except OSError:
        message = (
            f"Cloud SQL Auth Proxy is not listening on {CLOUDSQL_HOST}:{CLOUDSQL_PORT}. Start it first:"
            + "\n\n"
            + f"cloud-sql-proxy {CLOUDSQL_INSTANCE_CONNECTION_NAME} --port {CLOUDSQL_PORT}"
        )
        raise RuntimeError(message) from None



if RUN_CLOUDSQL_CATALOG:
    preflight_cloudsql()
    jdbc_uri = f"jdbc:postgresql://{CLOUDSQL_HOST}:{CLOUDSQL_PORT}/{CLOUDSQL_DB}"
    builder = (
        builder
        .config(f"spark.sql.catalog.{CLOUDSQL_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
        .config(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog")
        .config(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.uri", jdbc_uri)
        .config(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.jdbc.user", CLOUDSQL_USER)
        .config(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.jdbc.password", ensure_cloudsql_password())
        .config(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.jdbc.schema-version", "V1")
        .config(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.warehouse", WAREHOUSE)
    )

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"GCS catalog  : {GCS_ICEBERG_CATALOG}")
print(f"Cloud SQL cat: {CLOUDSQL_CATALOG if RUN_CLOUDSQL_CATALOG else 'disabled'}")


## 3. GCS Hadoop Iceberg Catalog

This section uses the GCS Hadoop catalog without Cloud SQL.

Important distinction:

- New exploration tables created by this Hadoop catalog can be read and written here.
- The existing production `alpaca.bars` table was created through the JDBC catalog. HadoopCatalog may list its directory, but loading it can fail if `metadata/version-hint.text` is absent. In that case, use the Cloud SQL catalog section for production data.


In [ ]:
if RUN_HADOOP_ICEBERG_READ:
    spark.sql(f"SHOW NAMESPACES IN {GCS_ICEBERG_CATALOG}").show(truncate=False)
    spark.sql(f"SHOW TABLES IN {GCS_ICEBERG_CATALOG}.{PROD_NAMESPACE}").show(truncate=False)
    print(f"Attempting to read {PROD_TABLE_ID} through HadoopCatalog")
    try:
        spark.sql(f"""
            SELECT S AS symbol, t, o, h, l, c, v
            FROM {PROD_TABLE_ID}
            ORDER BY t DESC
            LIMIT 20
        """).show(20, truncate=False)
    except Exception as exc:
        print("Production table could not be loaded through HadoopCatalog.")
        print("This is expected when the table was committed by the JDBC catalog and lacks metadata/version-hint.text.")
        print("Use the Cloud SQL catalog section below for production Iceberg semantics.")
        print(f"Error type: {type(exc).__name__}")
        print(str(exc).splitlines()[0])
else:
    print("Hadoop Iceberg read skipped. Set RUN_HADOOP_ICEBERG_READ=true to enable.")


## 4. GCS Hadoop Iceberg Write Smoke

This creates or replaces a small sandbox Iceberg table under the GCS Hadoop catalog. It is disabled by default.


In [ ]:
if RUN_HADOOP_ICEBERG_WRITE_SMOKE:
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {GCS_ICEBERG_CATALOG}.{EXPLORE_NAMESPACE}")
    spark.sql(f"""
        CREATE OR REPLACE TABLE {EXPLORE_ICEBERG_TABLE} (
            symbol STRING,
            t TIMESTAMP,
            close DOUBLE,
            source STRING
        )
        USING iceberg
        PARTITIONED BY (days(t))
    """)
    rows = [
        ("AAPL", "2026-07-19 10:00:00", 210.50, "notebook"),
        ("NVDA", "2026-07-19 10:01:00", 125.25, "notebook"),
        ("TSLA", "2026-07-19 10:02:00", 305.75, "notebook"),
    ]
    df = spark.createDataFrame(rows, "symbol string, t string, close double, source string").withColumn("t", F.to_timestamp("t"))
    df.writeTo(EXPLORE_ICEBERG_TABLE).append()
    spark.sql(f"SELECT * FROM {EXPLORE_ICEBERG_TABLE} ORDER BY t").show(truncate=False)
else:
    print("Iceberg write smoke skipped. Set RUN_HADOOP_ICEBERG_WRITE_SMOKE=true to create a sandbox table.")


## 5. Native Spark File Formats On GCS

Spark can read/write plain file formats independently of Iceberg. These paths do not have Iceberg table metadata.


In [ ]:
sample_df = spark.createDataFrame(
    [
        ("AAPL", "2026-07-19T10:00:00Z", 210.50, 100),
        ("NVDA", "2026-07-19T10:01:00Z", 125.25, 200),
        ("TSLA", "2026-07-19T10:02:00Z", 305.75, 300),
    ],
    "symbol string, t string, close double, volume long",
)

if RUN_NATIVE_FILE_WRITE_SMOKE:
    paths = {
        "parquet": f"{EXPLORE_FILE_ROOT}/parquet/sample_bars",
        "csv": f"{EXPLORE_FILE_ROOT}/csv/sample_bars",
        "json": f"{EXPLORE_FILE_ROOT}/json/sample_bars",
        "orc": f"{EXPLORE_FILE_ROOT}/orc/sample_bars",
    }
    sample_df.write.mode("overwrite").parquet(paths["parquet"])
    sample_df.write.mode("overwrite").option("header", "true").csv(paths["csv"])
    sample_df.write.mode("overwrite").json(paths["json"])
    sample_df.write.mode("overwrite").orc(paths["orc"])

    print("Parquet")
    spark.read.parquet(paths["parquet"]).show(truncate=False)
    print("CSV")
    spark.read.option("header", "true").csv(paths["csv"]).show(truncate=False)
    print("JSON")
    spark.read.json(paths["json"]).show(truncate=False)
    print("ORC")
    spark.read.orc(paths["orc"]).show(truncate=False)
else:
    print("Native file write smoke skipped. Set RUN_NATIVE_FILE_WRITE_SMOKE=true to write Parquet/CSV/JSON/ORC samples.")
    sample_df.show(truncate=False)


## 6. Optional Cloud SQL JDBC Iceberg Catalog

Enable this only when Cloud SQL is running and Cloud SQL Auth Proxy is listening locally.

Start proxy example:

```bash
cloud-sql-proxy project-66783f65-9c3e-4880-9a3:us-east1:alpaca-iceberg-catalog --port 5432
```

Then restart the kernel with `RUN_CLOUDSQL_CATALOG=true` in the environment or set the flag in the configuration cell before Spark starts.


In [ ]:
if RUN_CLOUDSQL_CATALOG:
    print(f"Reading {CLOUDSQL_TABLE_ID}")
    spark.sql(f"SHOW NAMESPACES IN {CLOUDSQL_CATALOG}").show(truncate=False)
    spark.sql(f"SHOW TABLES IN {CLOUDSQL_CATALOG}.{PROD_NAMESPACE}").show(truncate=False)
    spark.sql(f"""
        SELECT S AS symbol, COUNT(*) AS rows, MIN(t) AS first_t, MAX(t) AS latest_t
        FROM {CLOUDSQL_TABLE_ID}
        GROUP BY S
        ORDER BY rows DESC, symbol
        LIMIT 50
    """).show(50, truncate=False)
else:
    print("Cloud SQL catalog disabled. Set RUN_CLOUDSQL_CATALOG=true before starting Spark to enable.")


## 7. Stop Spark


In [ ]:
# Run when finished if you want to release local Spark resources.
# spark.stop()
